# 머신러닝 기반 영화 리뷰 감성 분석
1. 데이터 준비 (전처리된 데이터 -> 특징 벡터 추출 (TF-IDF))
2. 머신러닝 모델별 학습 및 평가
3. 배포 준비 : 영화 리뷰 긍부정 판단 테스트 -> 클래스 생성, 모델 저장


## 1.데이터 준비 (전처리된 데이터)

In [1]:
import pandas as pd

data_filename = './data/Korean_movie_reviews_2016.csv'

# 데이터 로딩
review_df = pd.read_csv(data_filename)
review_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력 데이터와 정답 데이터 추출
review_list = list(review_df.review)
label_list = list(review_df.label)

In [4]:
# 학습 데이터와 평가 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.2)
len(train_X), len(test_X), len(train_y), len(test_y)


(132307, 33077, 132307, 33077)

## 2. 특징 벡터 추출 : TF-IDF

In [56]:
# 한국어 감성 분석용 tokenizer 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

In [57]:
# 최대 단어 수 1000개
# 학습 데이터로 Vectorizer 생성 및 학습 데이터 특징 벡터 추출
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=korean_tokenizer, max_features=1000)
vectorizer.fit(train_X)
len(vectorizer.get_feature_names_out())



c:\Users\user01\anaconda3\envs\aiservice26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1000

In [7]:
# 학습 데이터의 특징 벡터 추출
train_X_fv = vectorizer.transform(train_X)
print(train_X_fv.shape)
print(train_X_fv)

(132307, 1000)
  (np.int32(0), np.int32(96))	0.40930872946715724
  (np.int32(0), np.int32(121))	0.383353969683605
  (np.int32(0), np.int32(394))	0.3192184327487526
  (np.int32(0), np.int32(493))	0.40407829227745123
  (np.int32(0), np.int32(584))	0.36539206345205294
  (np.int32(0), np.int32(683))	0.38676477453347297
  (np.int32(0), np.int32(819))	0.37044307654441955
  (np.int32(1), np.int32(175))	0.3351559950327157
  (np.int32(1), np.int32(228))	0.40683930618918507
  (np.int32(1), np.int32(310))	0.3659705181183687
  (np.int32(1), np.int32(720))	0.4476930941839445
  (np.int32(1), np.int32(728))	0.4655196625152398
  (np.int32(1), np.int32(820))	0.21762791885469615
  (np.int32(1), np.int32(823))	0.3517360430322468
  (np.int32(2), np.int32(2))	0.1762395364850962
  (np.int32(2), np.int32(36))	0.15132208026360078
  (np.int32(2), np.int32(48))	0.1963735525438505
  (np.int32(2), np.int32(62))	0.1962398074460766
  (np.int32(2), np.int32(63))	0.18539508024544016
  (np.int32(2), np.int32(86))	0.15

In [8]:
train_X_fv.toarray()[:1]

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [9]:
# 테스트 데이터 특징 벡터 추출
test_X_fv = vectorizer.transform(test_X)
print(test_X_fv)

  (np.int32(0), np.int32(86))	0.4382922864720345
  (np.int32(0), np.int32(144))	0.5675280658857308
  (np.int32(0), np.int32(430))	0.32225410842277563
  (np.int32(0), np.int32(473))	0.17873576560000212
  (np.int32(0), np.int32(619))	0.29306187382131277
  (np.int32(0), np.int32(752))	0.20162006520610573
  (np.int32(0), np.int32(793))	0.2163282089785719
  (np.int32(0), np.int32(794))	0.16879863786693194
  (np.int32(0), np.int32(962))	0.22604320935170683
  (np.int32(0), np.int32(983))	0.31160112729279343
  (np.int32(1), np.int32(17))	0.3897121905514439
  (np.int32(1), np.int32(621))	0.6291357809003189
  (np.int32(1), np.int32(622))	0.1980337481502972
  (np.int32(1), np.int32(781))	0.642724834062047
  (np.int32(2), np.int32(487))	0.5718930475803424
  (np.int32(2), np.int32(587))	0.512215044844603
  (np.int32(2), np.int32(622))	0.2131356257224607
  (np.int32(2), np.int32(682))	0.4118235535667771
  (np.int32(2), np.int32(960))	0.44220883724733123
  (np.int32(3), np.int32(315))	0.7347413387230

In [10]:
# 정답 데이터 변환 (np.array)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)


In [13]:
# 데이터 일부 확인
print(train_y[:10])
print(test_y[:10])


[0 0 0 1 1 1 0 1 0 1]
[0 1 1 1 1 0 0 1 0 1]


## 3. 머신러닝 모델별 학습 및 평가
* 의사결정 트리
* 랜덤포레스트
* 나이브 베이즈
* 로지스틱 회귀 분석
* SVM
* Perceptron

In [22]:
# 머신러닝 모델별 학습 성능 평가 결과 저장 준비
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])

## 3.1 의사결정 트리 (Decision Tree)

In [17]:
# 학습
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [20]:
# 성능 평가
train_score = dtc.score(train_X_fv, train_y) * 100
test_score = dtc.score(test_X_fv, test_y) * 100
train_score, test_score

(98.13993212755183, 79.847023611573)

In [23]:
# 평가 결과 score_df에 추가
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df


,train,test
DecisionTree,98.139932,79.847024


## 3.2 랜덤 포레스트 (Random Forrest)

In [24]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs = -1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [25]:
train_score = rf.score(train_X_fv, train_y) * 100
test_score = rf.score(test_X_fv, test_y) * 100
print(train_score, test_score)

98.13917630964349 84.75980288417934


In [26]:
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.139932,79.847024
RandomForest,98.139176,84.759803


## 3.3 나이브 베이즈 (Naive Baysian)

In [27]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [28]:
train_score = mnb.score(train_X_fv, train_y) * 100
test_score = mnb.score(test_X_fv, test_y) * 100
print(train_score, test_score)

85.31823713031056 85.13468573328899


In [29]:
score_df.loc['NaiveBayes'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.139932,79.847024
RandomForest,98.139176,84.759803
NaiveBayes,85.318237,85.134686


## 3.4 로지스틱 회귀 분석 (Logistic Regression)

In [31]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [32]:
train_score = lr.score(train_X_fv, train_y) * 100
test_score = lr.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.26452115156417 85.73328899235118


In [33]:
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.139932,79.847024
RandomForest,98.139176,84.759803
NaiveBayes,85.318237,85.134686
LogisticRegression,86.264521,85.733289


## 3.5 SVM (Support Vector Machine)

In [34]:
from sklearn.svm import LinearSVC
svc = LinearSVC(verbose=True)
svc.fit(train_X_fv, train_y)

[LibLinear]

LinearSVC(verbose=True)

In [35]:
train_score = svc.score(train_X_fv, train_y) * 100
test_score = svc.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.24789315758048 85.66073102155578


In [36]:
score_df.loc['SVM'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.139932,79.847024
RandomForest,98.139176,84.759803
NaiveBayes,85.318237,85.134686
LogisticRegression,86.264521,85.733289
SVM,86.247893,85.660731


## 3.6 Perceptron

## 3.7 성능 비교

In [38]:
# 평가 결과 저장 데이터 프레임 확인
score_df.sort_values(by='test', ascending=False)

,train,test
LogisticRegression,86.264521,85.733289
SVM,86.247893,85.660731
NaiveBayes,85.318237,85.134686
RandomForest,98.139176,84.759803
DecisionTree,98.139932,79.847024


## 4. 영화 리뷰 긍부정 판단
* 학습된 모델 중 선택하여 활용

In [41]:
# 영화 리뷰감성 분석용 tokenizer 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

# 특징 벡터 추출 모델 : vectorizor

# 학습 모델 선택
sa_model = lr

In [45]:
review = '영화가 재미있다'

# 텍스트 전처리

# 특징 벡터 추출
review_fv = vectorizer.transform([review])

# 예측
pred = sa_model.predict(review_fv)

# 예측 결과 출력
result = '긍정' if pred[0] >= 0.5 else '부정'
print(f'{review} -> {result}({pred[0]})')

영화가 재미있다 -> 긍정(1)


In [46]:
# 함수로 만들기
def analyze_semtiment(review):
    # 특징 벡터 추출
    review_fv = vectorizer.transform([review])
    # 학습된 모델로 예측
    pred = sa_model.predict(review_fv)
    # 예측값에 따라 결과를 생성하여 반환
    result = '긍정' if pred[0] >= 0.5 else '부정'
    return result, pred[0]

In [50]:
sa_model = svc

# 함수 테스트
reviews = [
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '하품만 나온다',
    '이 영화 핵노잼 ㅠㅠ',
    '이딴게 영화냐 ㅉㅉ',
    '와 개쩐다',
    '감독 뭐하는 놈이냐',
    '정말 세계관 최강자들의 영화다'
]

for review in reviews:
    sentiment, prob = analyze_semtiment(review)
    print(f'{review} -> {sentiment}({prob})')


이 영화 개꿀잼 ㅋㅋㅋ -> 부정(0)
하품만 나온다 -> 긍정(1)
이 영화 핵노잼 ㅠㅠ -> 부정(0)
이딴게 영화냐 ㅉㅉ -> 부정(0)
와 개쩐다 -> 긍정(1)
감독 뭐하는 놈이냐 -> 부정(0)
정말 세계관 최강자들의 영화다 -> 긍정(1)


In [51]:
# 문장을 입력 받아서 긍부정 판단 
review = input('>> 리뷰 입력 : ')
sentiment, prob = analyze_semtiment(review)
print(f'{review} -> {sentiment}({prob})')

영화를 보다가 졸았다 -> 부정(0)


In [58]:
# 배포를 위한 모델 저장

# 특징 추출용 vectorizer
import joblib
joblib.dump(vectorizer, './model/sa_movie_vectorizer.pkl')

# 감성 분석 모델 sa_model
joblib.dump(sa_model, './model/sa_movie_predict.pkl')

['./model/sa_movie_predict.pkl']

In [ ]:
import joblib

def korean_tokenizer(text):
        my_tags = ['Noun', 'Adjective', 'Verb']
        my_stopwords = []
        return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

class SentimentAnalyzer:
    def __init__(self, tokenizer, vectorizer_file, predict_model_file):
        self.__vecotrizer = joblib.load(vectorizer_file)
        self.__predict_model = joblib.load(predict_model_file)
        self.__tokenizer = tokenizer
    
    def analyze_semtiment(self, review):
        # 특징 벡터 추출
        review_fv = self.__vectorizer.transform([review])
        # 학습된 모델로 예측
        pred = self.__predict_model.predict(review_fv)
        # 예측값에 따라 결과를 생성하여 반환
        result = '긍정' if pred[0] >= 0.5 else '부정'
        return result, pred[0]
